### 1. ColumnTransformer and Pipeline (Dataset: IRIS)

In [34]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

data = load_iris(as_frame=True)
X = data.data
y = data.target

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_cols = ["sepal length (cm)", "sepal width (cm)",
                "petal length (cm)", "petal width (cm)"]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipe, numeric_cols),
])

prep_pipeline = Pipeline([
    ("preprocessor", preprocessor),
])

X_train_processed = prep_pipeline.fit_transform(x_train)
X_test_processed = prep_pipeline.transform(x_test)

feature_names = prep_pipeline.named_steps["preprocessor"].get_feature_names_out()
print("Processed train shape:", X_train_processed.shape)
print("Processed test shape:", X_test_processed.shape)
print("\nFeature names:", feature_names)
print("Total features:", len(feature_names))

Processed train shape: (120, 4)
Processed test shape: (30, 4)

Feature names: ['num__sepal length (cm)' 'num__sepal width (cm)' 'num__petal length (cm)'
 'num__petal width (cm)']
Total features: 4


### 3. Model Creation (XGBClassifier)

In [35]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from xgboost import XGBClassifier

model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", XGBClassifier(learning_rate = 0.1, max_depth = 3, n_estimators = 100)),
])

model.fit(x_train, y_train)

y_pred = model.predict(x_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("\nConfusion matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

Accuracy: 0.933

Confusion matrix:
 [[10  0  0]
 [ 0  9  1]
 [ 0  1  9]]

Classification report:
               precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       0.90      0.90      0.90        10
           2       0.90      0.90      0.90        10

    accuracy                           0.93        30
   macro avg       0.93      0.93      0.93        30
weighted avg       0.93      0.93      0.93        30



### 4. Flower Prediction

In [36]:
new_flower = pd.DataFrame(
    [[6.0, 2.9, 4.5, 1.5]],
    columns=numeric_cols
)

pred = model.predict(new_flower)[0]
proba = model.predict_proba(new_flower)[0]

print("Predicted flower:", data.target_names[pred])
for name, p in zip(data.target_names, proba):
    print(f"{name}: {p:.2f}")

Predicted flower: versicolor
setosa: 0.00
versicolor: 1.00
virginica: 0.00
